# Gold Rod Heat Diffusion — PINN vs. FEM

This notebook solves the 1D heat equation for a gold rod with:
- a heater fixed at `T_LEFT` at `x = 0` (Dirichlet)
- an **insulated** end at `x = L` (Neumann, zero flux)
- a cold rod initially at `T_INIT`

Two independent solvers are built and compared:
1. **PINN** — a neural network trained to satisfy the PDE + boundary/initial conditions via a loss function.
2. **FEM** — a classical finite element solver, used as ground truth to validate the PINN.

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.linalg import lu_factor, lu_solve

torch.manual_seed(0)
np.random.seed(0)

### Physical parameters

The right boundary is **insulated**, not held at a fixed cold temperature —
so there is no `T_RIGHT` value, just `T_INIT` and `T_LEFT`.

In [ ]:
L = 1.0          # rod length (m)
T_MAX = 60.0      # total simulated time (s)
alpha = 1.22e-3   # thermal diffusivity (m^2/s)

T_INIT = 20.0     # initial rod temperature (deg C), applied at t = 0 for all x
T_LEFT = 100.0    # boundary condition at x = 0, held for all t (Dirichlet / heater)
# x = L is insulated: zero heat flux (Neumann), no fixed value

## 2. PINN Solver

### 2.1 Model definition

Plain tanh MLPs are biased toward learning smooth, low-frequency functions
("spectral bias") -- which is why the earlier version collapsed to a
low-varying quadratic instead of capturing the sharp boundary layer near the
heater. A **Fourier feature embedding** gives the network direct access to
higher-frequency basis functions, which is the standard fix for this.

In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self, num_features=32, scale=2.0):
        super().__init__()
        # fixed (non-trainable) random projection -- this is what lets the
        # network represent sharp, high-frequency features
        self.B = nn.Parameter(torch.randn(2, num_features) * scale, requires_grad=False)

    def forward(self, xt):
        proj = xt @ self.B
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)


class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.fourier = FourierFeatures(num_features=32, scale=2.0)
        self.net = nn.Sequential(
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x, t):
        # normalize t to [0, 1] range internally so both inputs are comparable
        t_norm = t / T_MAX
        inp = torch.stack((x, t_norm), dim=1)
        feat = self.fourier(inp)
        return self.net(feat)


model = PINN()
model

### 2.2 Training configuration

In [ ]:
N_F = 2000    # interior (physics) points
N_IC = 200    # initial-condition points
N_BC = 200    # boundary points per side

adam_epochs = 3500
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

w_physics, w_ic, w_bc = 7.0, 1.0, 4.0

### 2.3 Stage 1 — Adam training loop

Key fixes baked in here:
- **Full time range restored.** Collocation points must be scaled by `T_MAX`
  and `L` -- a dropped `* T_MAX` previously meant the physics residual was
  only ever checked in the first ~1 second of the 60-second simulation,
  leaving the rest of the domain completely unconstrained.
- **Both boundary conditions enforced.** Left (`x=0`) is Dirichlet, fixed at
  `T_LEFT`. Right (`x=L`) is Neumann, enforced by penalizing `du/dx` there
  rather than pinning a value.
- **IC sampling excludes a tiny neighborhood of `x=0`**, since the true
  solution is discontinuous exactly at the `(x=0, t=0)` corner (heater
  switches on instantly while the rod starts cold). Fitting that single
  contradictory point blurs and drifts the boundary layer.
- **Collocation points are biased toward `x=0` and `t=0`**, where the
  solution has the sharp boundary layer, instead of sampled uniformly. This
  gives the sharp region proportionally more weight in the loss, since a
  uniform sample would spend most of its points in the boring, nearly-flat
  far field.

In [ ]:
loss_history = {"total": [], "physics": [], "ic": [], "bc": []}

for epoch in range(adam_epochs):
    optimizer.zero_grad()

    # ---- interior collocation points, biased toward x=0 and t=0 ----
    u_rand = torch.rand(N_F)
    x_f = (L * (1 - torch.sqrt(1 - u_rand))).clone().requires_grad_(True)
    t_f = ((torch.rand(N_F) ** 2) * T_MAX).clone().requires_grad_(True)

    u_f = model(x_f, t_f)
    du_dx = torch.autograd.grad(u_f, x_f, torch.ones_like(u_f), create_graph=True)[0]
    d2u_dx2 = torch.autograd.grad(du_dx, x_f, torch.ones_like(du_dx), create_graph=True)[0]
    du_dt = torch.autograd.grad(u_f, t_f, torch.ones_like(u_f), create_graph=True)[0]

    res = du_dt - alpha * d2u_dx2
    physics_loss = torch.mean(res ** 2)

    # ---- initial condition: exclude a tiny neighborhood of x=0 ----
    x_ic = torch.rand(N_IC) * L
    x_ic = x_ic[x_ic > 0.02 * L]
    t_ic = torch.zeros_like(x_ic)
    u_ic = model(x_ic, t_ic)
    ic_loss = torch.mean((u_ic - T_INIT) ** 2)

    # ---- left boundary: Dirichlet, heater fixed at T_LEFT ----
    x_bc1 = torch.zeros(N_BC)
    t_bc1 = torch.linspace(0, T_MAX, N_BC)
    bc1_loss = torch.mean((model(x_bc1, t_bc1) - T_LEFT) ** 2)

    # ---- right boundary: Neumann, insulated (zero flux -> du/dx = 0) ----
    x_bc2 = torch.full((N_BC,), L, requires_grad=True)
    t_bc2 = torch.linspace(0, T_MAX, N_BC)
    u_bc2 = model(x_bc2, t_bc2)
    du_dx_bc2 = torch.autograd.grad(u_bc2, x_bc2, torch.ones_like(u_bc2), create_graph=True)[0]
    bc2_loss = torch.mean(du_dx_bc2 ** 2)

    bc_loss = bc1_loss + bc2_loss

    # ---- total loss ----
    loss = w_physics * physics_loss + w_ic * ic_loss + w_bc * bc_loss
    loss.backward()
    optimizer.step()

    loss_history["total"].append(loss.item())
    loss_history["physics"].append(physics_loss.item())
    loss_history["ic"].append(ic_loss.item())
    loss_history["bc"].append(bc_loss.item())

    if epoch % 500 == 0:
        print(f"epoch {epoch:5d} | total {loss.item():.4f} | "
              f"physics {physics_loss.item():.4f} | ic {ic_loss.item():.4f} | "
              f"bc {bc_loss.item():.4f}")

### 2.3b Stage 2 — L-BFGS refinement

This runs **after** Adam finishes, as a separate phase -- not folded into
the epoch loop above. The two don't mix well:

- `lbfgs.step(closure)` is not "one step per epoch." Internally it calls
  `closure()` many times (`max_iter` below) while it performs its own line
  search, so it's a whole optimization run in a single call.
- LBFGS's internal line search assumes it's optimizing a fixed function.
  Resampling collocation points between its internal `closure()` calls
  (like Adam does every epoch) would shift the loss landscape mid-search,
  which risks divergence. So the points here are sampled **once** and held
  fixed for the whole L-BFGS run.
- Adam is good at cheaply getting into the right neighborhood; L-BFGS is
  good at precisely polishing once you're already close. Running L-BFGS
  first, or interleaving the two, wastes its precision on a bad starting
  point.

**Update:** an earlier version of this cell ran a single L-BFGS call with
`max_iter=500` against one frozen sample of points. That let the network
overfit that exact sample -- especially with the Fourier feature layer
supplying high-frequency basis functions to do it with -- producing
oscillatory, physically nonsensical output between the sampled points
(visible as wild swings between FEM and PINN curves, and values exceeding
the heater temperature). The fix below runs L-BFGS as several shorter
outer passes (50 iterations each) with fresh collocation points drawn
between passes, and a lower learning rate (`0.1` instead of `0.5`). Each
individual pass still uses one fixed sample during its own internal line
search, so it remains well-posed -- it just isn't allowed to grind on a
single sample for the full 500 iterations.

In [ ]:
# ---- L-BFGS run as several shorter outer passes, resampling collocation
# points between passes. A single very long L-BFGS run (many iterations
# against one frozen sample) let the network overfit that exact sample --
# especially with Fourier features supplying high-frequency basis functions
# to do it with -- producing oscillatory, physically nonsensical output
# between the sampled points. Shorter passes + fresh samples + a smaller
# learning rate keep it from happening again.

lbfgs_outer_passes = 10
lbfgs_iters_per_pass = 50

for outer in range(lbfgs_outer_passes):
    # fresh collocation points every pass
    u_rand = torch.rand(N_F)
    x_f_lbfgs = (L * (1 - torch.sqrt(1 - u_rand))).clone().requires_grad_(True)
    t_f_lbfgs = ((torch.rand(N_F) ** 2) * T_MAX).clone().requires_grad_(True)

    x_ic_lbfgs = torch.rand(N_IC) * L
    x_ic_lbfgs = x_ic_lbfgs[x_ic_lbfgs > 0.02 * L]
    t_ic_lbfgs = torch.zeros_like(x_ic_lbfgs)

    x_bc1_lbfgs = torch.zeros(N_BC)
    t_bc1_lbfgs = torch.linspace(0, T_MAX, N_BC)

    x_bc2_lbfgs = torch.full((N_BC,), L, requires_grad=True)
    t_bc2_lbfgs = torch.linspace(0, T_MAX, N_BC)

    lbfgs = torch.optim.LBFGS(model.parameters(), lr=0.1, max_iter=lbfgs_iters_per_pass,
                               history_size=50, line_search_fn='strong_wolfe')

    def closure():
        lbfgs.zero_grad()

        # ---- physics residual ----
        u_f = model(x_f_lbfgs, t_f_lbfgs)
        du_dx = torch.autograd.grad(u_f, x_f_lbfgs, torch.ones_like(u_f), create_graph=True)[0]
        d2u_dx2 = torch.autograd.grad(du_dx, x_f_lbfgs, torch.ones_like(du_dx), create_graph=True)[0]
        du_dt = torch.autograd.grad(u_f, t_f_lbfgs, torch.ones_like(u_f), create_graph=True)[0]
        res = du_dt - alpha * d2u_dx2
        physics_loss = torch.mean(res ** 2)

        # ---- initial condition ----
        u_ic = model(x_ic_lbfgs, t_ic_lbfgs)
        ic_loss = torch.mean((u_ic - T_INIT) ** 2)

        # ---- left boundary: Dirichlet ----
        bc1_loss = torch.mean((model(x_bc1_lbfgs, t_bc1_lbfgs) - T_LEFT) ** 2)

        # ---- right boundary: Neumann, zero flux ----
        u_bc2 = model(x_bc2_lbfgs, t_bc2_lbfgs)
        du_dx_bc2 = torch.autograd.grad(u_bc2, x_bc2_lbfgs, torch.ones_like(u_bc2), create_graph=True)[0]
        bc2_loss = torch.mean(du_dx_bc2 ** 2)

        bc_loss = bc1_loss + bc2_loss

        # ---- total loss ----
        loss = w_physics * physics_loss + w_ic * ic_loss + w_bc * bc_loss
        loss.backward()
        return loss

    final_loss = lbfgs.step(closure)
    print(f"L-BFGS pass {outer+1:2d}/{lbfgs_outer_passes} | loss {final_loss.item():.5f}")

### 2.4 Training diagnostics

In [ ]:
plt.figure(figsize=(7, 4))
for key, vals in loss_history.items():
    plt.plot(vals, label=key)
plt.yscale("log")
plt.xlabel("epoch (Adam phase)")
plt.ylabel("loss (log scale)")
plt.legend()
plt.title("Adam-phase training loss components (L-BFGS runs after, not logged per-epoch)")
plt.show()

### 2.5 PINN solution visualization

In [ ]:
n_plot = 120
x_lin = np.linspace(0, L, n_plot)
t_lin = np.linspace(0, T_MAX, n_plot)
x_grid, t_grid = np.meshgrid(x_lin, t_lin)

x_flat = torch.tensor(x_grid.flatten(), dtype=torch.float32)
t_flat = torch.tensor(t_grid.flatten(), dtype=torch.float32)

with torch.no_grad():
    u_grid = model(x_flat, t_flat).numpy().reshape(n_plot, n_plot)

print("u range:", u_grid.min(), "to", u_grid.max())

In [ ]:
fig = go.Figure(data=[go.Surface(
    x=x_lin,
    y=t_lin,
    z=u_grid,
    colorscale='Inferno',
    colorbar=dict(title='Temp (deg C)')
)])

fig.update_layout(
    title='PINN solution: temperature distribution in gold rod',
    scene=dict(
        xaxis_title='x (position along rod)',
        yaxis_title='t (time)',
        zaxis_title='u (temperature)',
        camera=dict(eye=dict(x=1.5, y=-1.8, z=0.9))
    ),
    width=800,
    height=600,
    margin=dict(l=0, r=0, t=40, b=0)
)

fig.show()

In [ ]:
plt.figure(figsize=(6, 5))
cp = plt.contourf(x_grid, t_grid, u_grid, levels=50, cmap='inferno')
plt.colorbar(cp, label='Temperature (deg C)')
plt.xlabel('x (position along rod)')
plt.ylabel('t (time)')
plt.title('PINN solution: temperature distribution in gold rod')
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
for time_val in range(0, 65, 5):
    position_tensor = torch.linspace(0, L, 100)
    time_tensor = torch.full_like(position_tensor, float(time_val), dtype=torch.float32)

    with torch.no_grad():
        temp_output = model(position_tensor, time_tensor)
        temp_values = temp_output.squeeze().cpu().numpy()

    plt.plot(position_tensor.cpu().numpy(), temp_values, label=f"t = {time_val}s")

plt.xlabel('x (position along rod)')
plt.ylabel('u (temperature)')
plt.title('PINN solution at several times')
plt.legend(fontsize=8, ncol=2)
plt.grid(True)
plt.show()

## 3. FEM Reference Solution

Classical finite-element solve of the same problem, used as ground truth.

- **Left (`x=0`):** Dirichlet — clamped exactly to `T_LEFT` every time step.
- **Right (`x=L`):** Neumann, zero flux — this is FEM's *natural* boundary
  condition. It requires no explicit enforcement: as long as the last row of
  the assembled system is left untouched, zero flux falls out automatically
  from the weak formulation.

In [ ]:
# ---- mesh ----
N_NODES = 101
n_dt = 6000
x_nodes = np.linspace(0, L, N_NODES)
h = x_nodes[1] - x_nodes[0]
dt = T_MAX / n_dt

# ---- assemble global mass (M) and stiffness (K) matrices ----
# standard linear (P1) 1D finite elements
M = np.zeros((N_NODES, N_NODES))
K = np.zeros((N_NODES, N_NODES))

M_local = (h / 6.0) * np.array([[2, 1],
                                 [1, 2]])
K_local = (1.0 / h) * np.array([[1, -1],
                                 [-1, 1]])

for e in range(N_NODES - 1):
    nodes = [e, e + 1]
    for a in range(2):
        for b in range(2):
            M[nodes[a], nodes[b]] += M_local[a, b]
            K[nodes[a], nodes[b]] += K_local[a, b]

# ---- time discretization: implicit (backward) Euler ----
# M * (T_new - T_old)/dt = -alpha * K * T_new
# => (M + alpha*dt*K) * T_new = M * T_old
A = M + alpha * dt * K

# ---- left boundary (x=0): Dirichlet, clamp exactly ----
A[0, :] = 0.0
A[0, 0] = 1.0

# ---- right boundary (x=L): Neumann, zero flux ----
# Natural boundary condition -- no modification needed. Leaving row
# N_NODES-1 of A untouched is what makes it insulated.

A_factored = lu_factor(A)

# ---- initial condition ----
T = np.full(N_NODES, T_INIT)
T[0] = T_LEFT     # heater is already on at t=0

# ---- store snapshots for plotting ----
snapshot_times = list(range(0, int(T_MAX) + 1, 5))
snapshots = {}
t_current = 0.0
next_snapshot_idx = 0

if 0 in snapshot_times:
    snapshots[0] = T.copy()
    next_snapshot_idx = 1

for step in range(n_dt):
    rhs = M @ T
    rhs[0] = T_LEFT      # only the left row needs overwriting
    # rhs[-1] left untouched -- no boundary value to impose

    T = lu_solve(A_factored, rhs)
    t_current += dt

    if (next_snapshot_idx < len(snapshot_times) and
            t_current >= snapshot_times[next_snapshot_idx] - 1e-9):
        snapshots[snapshot_times[next_snapshot_idx]] = T.copy()
        next_snapshot_idx += 1

plt.figure(figsize=(7, 5))
for t_val, T_snap in snapshots.items():
    plt.plot(x_nodes, T_snap, label=f"t = {t_val}s")

plt.xlabel("x (position along rod)")
plt.ylabel("Temperature (deg C)")
plt.title("FEM solution: insulated right end (Neumann BC)")
plt.legend(fontsize=8, ncol=2)
plt.grid(True)
plt.show()

## 4. Comparison: PINN vs. FEM

In [ ]:
plt.figure(figsize=(8, 6))
colors = plt.cm.inferno(np.linspace(0.15, 0.85, len(snapshots)))

for color, (t_val, T_snap) in zip(colors, snapshots.items()):
    plt.plot(x_nodes, T_snap, color=color, label=f"t = {t_val}s (FEM)")

    pinn_points = torch.tensor(x_nodes, dtype=torch.float32)
    pinn_time = torch.full_like(pinn_points, float(t_val), dtype=torch.float32)
    with torch.no_grad():
        pinn_temp = model(pinn_points, pinn_time).squeeze().numpy()
    plt.plot(x_nodes, pinn_temp, '--', color=color)

plt.plot([], [], 'k-', label='FEM (solid)')
plt.plot([], [], 'k--', label='PINN (dashed)')

plt.xlabel("x (position along rod)")
plt.ylabel("Temperature (deg C)")
plt.title("FEM (solid) vs. PINN (dashed) — insulated right end")
plt.legend(fontsize=8, ncol=2)
plt.grid(True)
plt.show()

In [ ]:
# ---- pointwise error over the full (x, t) grid ----
fem_grid = np.zeros_like(u_grid)
T_fem = np.full(N_NODES, T_INIT)
T_fem[0] = T_LEFT
t_current = 0.0
next_col = 0

fem_checkpoints = {}
if abs(t_lin[0] - 0.0) < 1e-9:
    fem_checkpoints[0] = T_fem.copy()
    next_col = 1

for step in range(n_dt):
    rhs = M @ T_fem
    rhs[0] = T_LEFT
    T_fem = lu_solve(A_factored, rhs)
    t_current += dt

    while next_col < len(t_lin) and t_current >= t_lin[next_col] - 1e-9:
        fem_checkpoints[next_col] = T_fem.copy()
        next_col += 1

for col_idx, T_fem_snap in fem_checkpoints.items():
    fem_grid[col_idx, :] = np.interp(x_lin, x_nodes, T_fem_snap)

error_grid = np.abs(u_grid - fem_grid)

plt.figure(figsize=(6, 5))
cp = plt.contourf(x_grid, t_grid, error_grid, levels=50, cmap='viridis')
plt.colorbar(cp, label='|PINN - FEM| (deg C)')
plt.xlabel('x (position along rod)')
plt.ylabel('t (time)')
plt.title('Pointwise absolute error: PINN vs. FEM')
plt.show()

print(f"Max error: {error_grid.max():.3f} deg C")
print(f"Mean error: {error_grid.mean():.3f} deg C")